# A5 — L3 Estate Simulator

Builds a parameterised financial-institution estate and lands **real** L1/L2 findings on it. Logic lives in `src/simulator/` (CLAUDE.md rule 9); every parameter is in `src/simulator/config.py` (rule 7) and is also written out to `data/simulator/l3_simulator_parameters.csv` for provenance.

Pipeline: bank structure → assets → IAM graph → inject real findings → lifecycle (SLA, recurrence) → label.

**What is real and what is simulated.** Every finding points at a real CVE (NVD/EPSS/KEV) or a real check (Prowler/Checkov/ScoutSuite). What is simulated is *which asset a real finding landed on*, plus the estate context (criticality tier, exposure, regulatory scope, SLA, recurrence).

**The label is a formula, not ground truth** — 60% real signal + 40% simulated context, fixed in CLAUDE.md. It must never be presented the way CISA KEV is, and L3 results are reported separately from L1/L2 (rule 12). A model scoring well here is partly recovering a deterministic function of its own inputs; the informative L3 experiments are about *robustness* (label noise, weight sweeps, ablation), not raw accuracy.

> ⚠️ **Open issue before Study C** — the formula puts L1 and L2 on incommensurate scales: the L1 real component has median 0.001 against L2's 0.500, so 99.4% of vulnerability findings come out `Low` and priority is effectively "is this a misconfiguration?". See the A5 entry in CLAUDE.md's Deviations Log. Needs a supervisor decision; not changed unilaterally.

In [1]:
import os
import sys
from pathlib import Path

# Every src/ path is relative to the project root (data/..., outputs/...).
# Without this chdir a run from notebooks/ writes into notebooks/data/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.simulator.build import build, log_stats

In [2]:
results = build()
log_stats(results)

2026-11-23 17:52:30,513 INFO src.simulator.build: Loading real L1/L2 inputs


2026-11-23 17:52:31,026 INFO src.simulator.assets: Generated 5000 assets (8.4% internet-facing, 39.9% in regulatory scope)


2026-11-23 17:52:31,047 INFO src.simulator.iam: Built IAM graph: 1100 principals, 5000 assets, 12020 edges


2026-11-23 17:52:31,070 INFO src.simulator.iam: IAM features: median blast radius 37 assets, 36.2% of assets on an admin path


2026-11-23 17:52:31,158 INFO src.simulator.injector: Vulnerability pool: 230198 CVEs published since 2022-11-24


2026-11-23 17:52:31,176 INFO src.simulator.injector: Injected 39989 findings (27538 misconfiguration, 12451 vulnerability) across 5000 assets


2026-11-23 17:52:31,198 INFO src.simulator.temporal: Lifecycle: 31.1% open, 30.0% SLA-breached, mean recurrence 0.16


2026-11-23 17:52:31,217 INFO src.simulator.labeller: Labels: Low 54.1%, Medium 40.3%, High 5.2%, Critical 0.4%


2026/11/23 17:52:32 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/spopa/Documents/Internship Project/internship-project/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026-11-23 17:52:33,107 INFO src.tracking: Logged A5 to MLflow (run 8403824862c9433ead776c7bcee5df88)



A5 L3 estate simulator
Assets:                  5,000
Findings:                39,989
  from real CVEs (L1):   12,451
  from real checks (L2): 27,538
Distinct real CVEs used: 11,999
Distinct real checks:    2,349

Estate shape:
  internet-facing:       8.4% of assets
  in regulatory scope:   39.9% of assets
  median blast radius:   37 assets
  on an admin path:      36.2% of assets

Criticality tiers (0 = most critical):
  tier 0:    201 (4.0%)
  tier 1:    855 (17.1%)
  tier 2:  2,826 (56.5%)
  tier 3:    980 (19.6%)
  tier 4:    138 (2.8%)

Lifecycle:
  open:                  31.1%
  SLA breached:          30.0%
  mean recurrences:      0.16

Priority label distribution (60% real / 40% simulated context):
  Critical      143 (0.4%)
  High        2,087 (5.2%)
  Medium     16,121 (40.3%)
  Low        21,638 (54.1%)

  priority_score: mean 0.269, p95 0.429, max 0.699
  label noise level: 0.0

REMINDER: this label is a documented formula, not ground truth. Report
L3 results separately f

## The estate

In [3]:
assets = results["assets"]
display(assets.head())
display(
    assets.groupby("business_unit").agg(
        assets=("asset_id", "count"),
        mean_tier=("criticality_tier", "mean"),
        internet_facing=("internet_facing", "mean"),
        regulated=("regulatory_scope", "mean"),
        median_blast_radius=("blast_radius", "median"),
    ).round(3)
)

,asset_id,business_unit,environment,provider,region,asset_type,criticality_tier,internet_facing,pci_scope,dora_scope,regulatory_scope,principal_count,admin_principal_count,blast_radius,has_admin_path
0,asset-000000,Corporate IT,production,azure,eu-west-1,serverless_function,3,False,False,False,False,3,0,37,False
1,asset-000001,Investment Banking,development,azure,eu-central-1,data_warehouse,2,False,False,False,False,0,0,0,False
2,asset-000002,Corporate IT,development,aws,eu-central-1,compute_instance,4,True,False,False,False,2,1,54,True
3,asset-000003,Risk & Compliance,production,aws,us-east-1,container_workload,2,False,False,True,True,6,2,154,True
4,asset-000004,Payments,production,aws,eu-west-1,object_storage,0,False,True,True,True,0,0,0,False


,assets,mean_tier,internet_facing,regulated,median_blast_radius
business_unit,,,,,
Corporate IT,787,2.691,0.076,0.203,36.0
Digital Channels,398,2.070,0.098,0.382,77.5
Investment Banking,913,1.785,0.084,0.432,26.0
Payments,705,1.655,0.089,0.444,33.0
Retail Banking,1136,1.898,0.082,0.463,21.0
Risk & Compliance,472,1.862,0.089,0.462,72.0
Wealth Management,589,2.080,0.081,0.396,57.0


## Does the label respond to its inputs?

A sanity check on the formula: every context term should move the score in the expected direction.

In [4]:
findings = results["findings"]
for column in ["criticality_tier", "internet_facing", "regulatory_scope", "sla_breached"]:
    print(f"--- mean priority_score by {column} ---")
    print(findings.groupby(column).priority_score.mean().round(3).to_string())
    print()

--- mean priority_score by criticality_tier ---
criticality_tier
0    0.370
1    0.339
2    0.254
3    0.240
4    0.212

--- mean priority_score by internet_facing ---
internet_facing
False    0.260
True     0.364

--- mean priority_score by regulatory_scope ---
regulatory_scope
False    0.228
True     0.331

--- mean priority_score by sla_breached ---
sla_breached
False    0.255
True     0.302



## The scale problem, shown directly

This is the open issue flagged above — the two layers are not on the same scale.

In [5]:
import pandas as pd

display(
    findings.groupby("source_layer").real_component.describe()[["mean", "50%", "max"]].round(3)
)
display(pd.crosstab(findings.source_layer, findings.priority_label, normalize="index").round(3))

,mean,50%,max
source_layer,,,
L1,0.139,0.108,0.993
L2,0.301,0.275,0.550


priority_label,Critical,High,Low,Medium
source_layer,,,,
L1,0.003,0.028,0.815,0.155
L2,0.004,0.063,0.417,0.516


## Grouped-split readiness

CLAUDE.md requires L3 to be split grouped-by-asset so the same asset never appears in both train and test. `asset_id` is carried on every finding for exactly this.

In [6]:
print("findings:", len(findings))
print("assets   :", findings.asset_id.nunique())
print("findings per asset:", round(len(findings) / findings.asset_id.nunique(), 2))

findings: 39989
assets   : 4999
findings per asset: 8.0
